# PCW Lesson 4: Trees 1 - Decision Trees and Gini Impurity

**Student**: Katia Gwaneza Nkurunziza  
**Date**: Session 4, Fall 2026  
**Topics**: Decision trees, Gini impurity, tree instability, axis-aligned splits

Most real-world machine learning happens on tabular data (spreadsheets), and decision trees are still the most commonly used and often best-performing models on this kind of data. This session covers how trees split data using Gini impurity, and why trees can be surprisingly unstable.

## LLM Prompt Used

> I'm an undergraduate computer science student studying for my introductory machine learning class on decision trees. I want you to act as a tutor for this class. Give me a quick rundown on how decision trees work. Please be sure to include an explanation of how Gini impurity can be used to split trees. Format your response as a 2-pager with bullet points.

**Summary of LLM Response:**

- A decision tree splits data repeatedly into smaller groups based on feature values, forming a tree of yes/no questions
- Each split tries to make child nodes more "pure" (mostly one class) than the parent node
- Gini impurity measures how mixed a node is: 0 = perfectly pure (one class only), 0.5 = maximally mixed (50/50 for binary classification)
- Formula: Gini = 1 - sum(p_i^2) across classes, where p_i is the fraction of class i in the node
- To pick a split, the tree tries every feature and every possible threshold, calculates the weighted Gini impurity of the resulting two child nodes, and picks whichever split gives the lowest weighted Gini impurity
- This process repeats recursively on each child node until a stopping condition (max depth, minimum samples, or pure nodes) is reached

## Setup

In [ ]:
import numpy as np
from sklearn import metrics, inspection, datasets
from matplotlib import pyplot as plt

np.random.seed(42)

## Question 1 of 5: Basic Questions for Class

Reading: Wilber, J., & Santamaría, L. (n.d.) - covers "The Problem of Perturbations" in decision trees.

### 1a. Why do small perturbations result in such massive differences in the splits of the tree?

Decision trees are built greedily: at each node, the algorithm picks whichever single split minimizes Gini impurity *at that node only*. It never looks ahead to see if a different split would produce a better tree overall.

Because of this greedy, recursive structure, trees are extremely sensitive to which split gets chosen first:

- If two candidate splits have nearly identical Gini impurity, a tiny change in the data (one point moving, one point added/removed) can flip which split "wins"
- Once the first split changes, every downstream split changes too, because each child node now contains a different subset of points
- This is a **cascading effect**: an early decision determines the entire shape of the tree beneath it. A linear model doesn't have this problem because moving one point only nudges the coefficients slightly; a tree's decision is discrete (a threshold either includes a point or it doesn't), so the effect is all-or-nothing at each split.

### 1b. Why is this a bad thing? If we have incredibly fine splits that fit our training data, isn't this desirable?

Fine splits that perfectly separate the training data are a symptom of **overfitting**, not good learning.

- A tree that keeps splitting until every leaf is pure has learned the *exact* arrangement of training points, including noise and outliers specific to that sample
- This is undesirable because the goal of ML is to generalize to new data, not memorize old data
- The instability itself is the diagnostic: if retraining on a slightly perturbed version of the same data produces a *completely different* tree (different splits, different depth) but *similar training accuracy*, that's a sign the tree isn't finding a stable, real signal. It's fitting whatever noise happens to be in front of it.
- In bias-variance terms: an unconstrained tree is **low bias, high variance**. It fits training data almost perfectly (low bias) but changes wildly with small data changes (high variance), which usually hurts test performance.

### 1c. What would the data splits look like when projected down into 2D?

Every decision tree split only looks at **one feature at a time** (an axis in feature space) and picks a threshold on that axis. Because of this:

- In 2D, every split is a straight line that is either perfectly horizontal or perfectly vertical (axis-aligned). A tree can never draw a diagonal or curved boundary directly
- The full set of splits partitions 2D space into a grid of rectangles (like a checkerboard with unevenly sized cells)
- For data with a genuinely curved boundary (like the two interleaving moons we use below), a tree can only *approximate* the curve with a staircase of small rectangular steps
- This means: to fit a smooth curve well, the tree needs many splits (deep tree), which directly causes the fine-splitting / overfitting problem from 1b

### Reflection: Poking Holes in the LLM's Explanation

The LLM's rundown on Gini impurity and splitting was mechanically correct, but it initially undersold **why** trees are unstable. It described the splitting algorithm as if it were a stable, deterministic search, without flagging that:

- Ties or near-ties between candidate splits are common and are where perturbation sensitivity actually lives
- The greedy, non-lookahead nature of the algorithm is the root cause of both the instability and the axis-aligned limitation, not just an implementation detail

When pushed with the follow-up questions above, the LLM correctly connected greediness to instability, but a first-pass tutor answer left this out. This matches a general pattern: default explanations describe the mechanics, but not the failure modes, until directly asked.

## Question 2 of 5: Core Questions - Visualize the Data

Generate the two-moons dataset and decide, just by looking at the plot, whether the first split should be on the X or Y axis, and roughly where.

In [ ]:
x_data, y_data = datasets.make_moons(n_samples=500, noise=0.15, random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(x_data[:, 0], x_data[:, 1], c=y_data, cmap='coolwarm', alpha=0.7, edgecolor='k', linewidth=0.3)
plt.xlabel('X (axis 0)')
plt.ylabel('Y (axis 1)')
plt.title('Two Moons Dataset')
plt.grid(alpha=0.3)
plt.show()

### Visual Prediction

Looking at the plot: the two moons are offset vertically (one shifted up, one shifted down) more cleanly than they are separated left-right. A horizontal line roughly through the middle of the Y-range looks like it would separate a good chunk of the top moon from the bottom moon, so my prediction is: **first split on the Y axis (axis 1), somewhere near the middle of the Y range** (roughly Y ≈ 0.1 to 0.2, since the moons are centered around Y ≈ -0.5 to 1.0 and cross over near the middle).

We'll verify this with the actual Gini impurity calculation in Question 3.

## Question 3 of 5: Find the Best First Split Using Gini Impurity

Using the provided Gini impurity code, test every unique threshold value on both axes and find the split that minimizes weighted Gini impurity.

In [ ]:
def leaf_gini_impurity(leaf_ys):
    if len(leaf_ys) == 0:
        return 0
    else:
        return 1 - (sum(leaf_ys==0)/len(leaf_ys))**2 - (sum(leaf_ys==1)/len(leaf_ys))**2

def weighted_gini_impurity(left_ys, right_ys):
    total = len(left_ys) + len(right_ys)
    left_leaf = leaf_gini_impurity(left_ys) * len(left_ys)/total
    right_leaf = leaf_gini_impurity(right_ys) * len(right_ys)/total
    return left_leaf + right_leaf

def gini_impurity(x_data, y_data, axis, threshold):
    # The axis argument should be 0 or 1
    # The threshold should be a real valued number
    left_mask = x_data[:, axis] <= threshold
    right_mask = x_data[:, axis] > threshold
    return weighted_gini_impurity(y_data[left_mask], y_data[right_mask])

# Sample usage:
# gini_impurity(x_data, y_data, axis=0, threshold=0.8)

In [ ]:
# Search every unique threshold on both axes for the lowest weighted Gini impurity

def find_best_split(x_data, y_data):
    best_gini = float('inf')
    best_axis = None
    best_threshold = None

    for axis in [0, 1]:
        thresholds = np.unique(x_data[:, axis])
        for t in thresholds:
            g = gini_impurity(x_data, y_data, axis, t)
            if g < best_gini:
                best_gini = g
                best_axis = axis
                best_threshold = t

    return best_axis, best_threshold, best_gini

initial_gini = leaf_gini_impurity(y_data)
print(f"Impurity before any split: {initial_gini:.4f}")

best_axis, best_threshold, best_gini = find_best_split(x_data, y_data)
axis_name = "X" if best_axis == 0 else "Y"
print(f"\nBest first split: axis={axis_name} (axis {best_axis}), threshold={best_threshold:.4f}")
print(f"Weighted Gini impurity after split: {best_gini:.4f}")
print(f"Improvement: {initial_gini - best_gini:.4f}")

### Result

**Best first split: Y axis (axis 1), threshold ≈ 0.147, weighted Gini impurity ≈ 0.263**

This confirms the visual prediction from Question 2: the tree splits on Y first, close to the middle of the Y range where the two moons cross over. The impurity drops from 0.500 (maximally mixed, since classes are balanced 50/50 before any split) down to about 0.263, which is a large improvement for a single split.

Let's visualize this split on the data:

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(x_data[:, 0], x_data[:, 1], c=y_data, cmap='coolwarm', alpha=0.7, edgecolor='k', linewidth=0.3)
plt.axhline(y=best_threshold, color='green', linewidth=2, linestyle='--', label=f'First split: Y={best_threshold:.3f}')
plt.xlabel('X (axis 0)')
plt.ylabel('Y (axis 1)')
plt.title('First Split (Root Node)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Question 4 of 5: Extension - Find the Second-Level Splits

Now that the first split divides the space into two halves (Y <= 0.147 and Y > 0.147), find the best split for each half independently.

In [ ]:
# Split the data into the two halves created by the first split
left_mask = x_data[:, best_axis] <= best_threshold
right_mask = x_data[:, best_axis] > best_threshold

x_left, y_left = x_data[left_mask], y_data[left_mask]
x_right, y_right = x_data[right_mask], y_data[right_mask]

print(f"Left node (Y <= {best_threshold:.4f}): {len(y_left)} samples, impurity = {leaf_gini_impurity(y_left):.4f}")
print(f"Right node (Y > {best_threshold:.4f}): {len(y_right)} samples, impurity = {leaf_gini_impurity(y_right):.4f}")

# Find the best split independently within each half
left_axis, left_threshold, left_gini = find_best_split(x_left, y_left)
right_axis, right_threshold, right_gini = find_best_split(x_right, y_right)

left_axis_name = "X" if left_axis == 0 else "Y"
right_axis_name = "X" if right_axis == 0 else "Y"

print(f"\n=== LEFT NODE SPLIT ===")
print(f"Axis: {left_axis_name} (axis {left_axis})")
print(f"Threshold: {left_threshold:.4f}")
print(f"Resulting weighted Gini impurity: {left_gini:.4f}")
print(f"Improvement from {leaf_gini_impurity(y_left):.4f} to {left_gini:.4f}")

print(f"\n=== RIGHT NODE SPLIT ===")
print(f"Axis: {right_axis_name} (axis {right_axis})")
print(f"Threshold: {right_threshold:.4f}")
print(f"Resulting weighted Gini impurity: {right_gini:.4f}")
print(f"Improvement from {leaf_gini_impurity(y_right):.4f} to {right_gini:.4f}")

In [ ]:
# Visualize all splits down to depth 2
plt.figure(figsize=(8, 6))
plt.scatter(x_data[:, 0], x_data[:, 1], c=y_data, cmap='coolwarm', alpha=0.7, edgecolor='k', linewidth=0.3)

# First split: horizontal line across full X range
plt.axhline(y=best_threshold, color='green', linewidth=2, linestyle='--', label=f'Split 1 (root): Y={best_threshold:.3f}')

# Second split for left node (Y <= threshold): vertical line, only within that Y range
x_range = [x_data[:, 0].min() - 0.2, x_data[:, 0].max() + 0.2]
y_range = [x_data[:, 1].min() - 0.2, x_data[:, 1].max() + 0.2]

if left_axis == 0:
    plt.plot([left_threshold, left_threshold], [y_range[0], best_threshold], color='purple', linewidth=2, linestyle='--', label=f'Split 2 (left node): X={left_threshold:.3f}')
else:
    plt.plot(x_range, [left_threshold, left_threshold], color='purple', linewidth=2, linestyle='--', label=f'Split 2 (left node): Y={left_threshold:.3f}')

# Second split for right node (Y > threshold): vertical line, only within that Y range
if right_axis == 0:
    plt.plot([right_threshold, right_threshold], [best_threshold, y_range[1]], color='orange', linewidth=2, linestyle='--', label=f'Split 2 (right node): X={right_threshold:.3f}')
else:
    plt.plot(x_range, [right_threshold, right_threshold], color='orange', linewidth=2, linestyle='--', label=f'Split 2 (right node): Y={right_threshold:.3f}')

plt.xlabel('X (axis 0)')
plt.ylabel('Y (axis 1)')
plt.title('Depth-2 Decision Tree Splits (Root + Two Children)')
plt.legend(loc='upper left', fontsize=9)
plt.grid(alpha=0.3)
plt.show()

### Summary of Results

| Level | Node | Axis | Threshold | Weighted Gini | Samples |
|-------|------|------|-----------|----------------|---------|
| 0 (root) | all data | Y (axis 1) | 0.1470 | 0.2627 | 500 |
| 1 | left (Y ≤ 0.147) | X (axis 0) | -0.7804 | 0.0731 | 210 |
| 1 | right (Y > 0.147) | X (axis 0) | 1.2375 | 0.1830 | 290 |

**What this shows**: the root split picked the Y axis because it gave the single biggest impurity reduction across the whole dataset. But once the data is divided, both second-level splits pick the **X axis** instead, because within each half, the remaining class mixing is now separated left-to-right rather than up-down. This is exactly the axis-aligned, staircase-like approximation of the curved moon boundary described in Question 1c: first a horizontal cut, then vertical cuts within each half, gradually carving out a jagged approximation of the two crescent shapes.

It also illustrates the perturbation sensitivity from Question 1a: the root split and both second-level splits were decisively better than their next-best alternative in this run (nontrivial gap in Gini scores), so this particular tree shape is fairly stable for this dataset. But with noisier data or fewer samples, splits with closer Gini scores would make the chosen axis/threshold flip easily between runs, producing a different-looking tree with similar overall accuracy.